# 08. External Validation — CheXpert (Domain Shift 분석)

## 목적
NIH ChestX-ray14로 학습한 모델을 **Stanford CheXpert** 데이터셋에 적용해  
도메인 시프트(Domain Shift) 크기를 정량화합니다.

## NIH ↔ CheXpert 공통 클래스 (7개)
| NIH 레이블 | CheXpert 컬럼 |
|-----------|---------------|
| Atelectasis | Atelectasis |
| Cardiomegaly | Cardiomegaly |
| Consolidation | Consolidation |
| Edema | Edema |
| Effusion | Pleural Effusion |
| Pneumonia | Pneumonia |
| Pneumothorax | Pneumothorax |

## Uncertain(-1) 처리
기본: **U-zeros** (−1 → 0, 보수적 평가)

In [ ]:
# ── 0. 환경 설정 ──────────────────────────────────────────────────────────────
import os, sys, subprocess

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    REPO_DIR = '/kaggle/working/CXR-CAD'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/sogang-cxr-cad/CXR-CAD', REPO_DIR], check=True)
    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)
    subprocess.run(['pip', 'install', '-q', 'timm', 'pyyaml'], check=True)
else:
    sys.path.insert(0, '..')
    os.chdir('..')

import yaml, torch
from pathlib import Path
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# ── 1. 경로 설정 ──────────────────────────────────────────────────────────────
with open('configs/config.yaml') as f:
    CFG = yaml.safe_load(f)

# CheXpert 루트 경로 (환경변수 > Kaggle 설정 > 수동 설정)
if os.environ.get('CHEXPERT_DIR'):
    CHEXPERT_ROOT = os.environ['CHEXPERT_DIR']
elif IS_KAGGLE:
    CHEXPERT_ROOT = CFG['kaggle']['chexpert_dir']
else:
    # 로컬 개발 시 직접 지정
    CHEXPERT_ROOT = 'data/chexpert/CheXpert-v1.0-small'  # ← 실제 경로로 변경

# NIH Checkpoint 경로
if IS_KAGGLE:
    # 04_Training 노트북 출력 데이터셋으로 연결하거나 경로 직접 지정
    CHECKPOINT_DIR = Path('/kaggle/input/cxr-cad-checkpoints')  # ← 실제 output slug
else:
    CHECKPOINT_DIR = Path(CFG['train']['checkpoint_dir'])

# 학습 설정
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = CFG['train']['batch_size']
N_WORKERS  = CFG['train']['num_workers']
USE_AMP    = CFG['train'].get('use_amp', True) and torch.cuda.is_available()

print(f'CheXpert  : {CHEXPERT_ROOT}')
print(f'Checkpoint: {CHECKPOINT_DIR}')
print(f'Device    : {DEVICE}')

In [ ]:
# ── 2. 임포트 ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from src.preprocess.chexpert_loader import (
    load_chexpert_csv, build_chexpert_val_loader,
    EVAL_LABELS, CHEXPERT_TO_NIH,
)
from src.preprocess.transforms import get_inference_transforms
from src.train.models import build_model, DISEASE_LABELS
from src.analysis.evaluation import compute_auroc, compute_auprc
from src.analysis.external_val import compare_internal_external

IMAGE_SIZE = CFG['data']['image_size']
UNCERTAIN  = CFG['chexpert']['uncertain_strategy']  # 'u_zeros'
EVAL_SPLIT = CFG['chexpert']['eval_split']          # 'valid'

In [ ]:
# ── 3. CheXpert 데이터 로드 ───────────────────────────────────────────────────
chex_loader, eval_labels = build_chexpert_val_loader(
    chexpert_root      = CHEXPERT_ROOT,
    split              = EVAL_SPLIT,
    uncertain_strategy = UNCERTAIN,
    transform          = get_inference_transforms(IMAGE_SIZE),
    batch_size         = BATCH_SIZE,
    num_workers        = N_WORKERS,
)

print(f'\nEval labels ({len(eval_labels)}개): {eval_labels}')
print(f'Dataloader batches: {len(chex_loader)}')

In [ ]:
# ── 4. 추론 함수 ──────────────────────────────────────────────────────────────
@torch.no_grad()
def run_inference(model, loader, device):
    """모델로 CheXpert 추론 → (y_true, y_prob) 반환 (shape: (N, 14))."""
    model.eval()
    all_probs, all_targets = [], []
    for images, labels in tqdm(loader, desc='CheXpert Inference'):
        images = images.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy()  # (B, 14)
        all_probs.append(probs)
        all_targets.append(labels.numpy())
    return (
        np.concatenate(all_targets, axis=0),  # y_true (N, 14)
        np.concatenate(all_probs,   axis=0),  # y_prob  (N, 14)
    )

In [ ]:
# ── 5. 모델별 CheXpert 추론 및 AUROC 계산 ────────────────────────────────────
# eval_indices: DISEASE_LABELS 내 EVAL_LABELS(7개) 위치
from src.preprocess.chexpert_loader import EVAL_LABELS
eval_indices = [DISEASE_LABELS.index(l) for l in EVAL_LABELS]

chex_results = {}  # {model_key: {label: auroc}}

for model_key in ['densenet', 'efficientnet', 'vit']:
    ckpt_path = CHECKPOINT_DIR / f'{model_key}_best.pth'
    if not ckpt_path.exists():
        print(f'[{model_key}] checkpoint 없음, 건너뜀')
        continue

    print(f'\n[{model_key}] CheXpert 추론 중...')
    model = build_model(model_key)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE)

    y_true_14, y_prob_14 = run_inference(model, chex_loader, DEVICE)

    # 7개 공통 클래스만 추출해 AUROC 계산
    y_true_7 = y_true_14[:, eval_indices]
    y_prob_7 = y_prob_14[:, eval_indices]
    auroc_dict = compute_auroc(y_true_7, y_prob_7, EVAL_LABELS)
    auprc_dict = compute_auprc(y_true_7, y_prob_7, EVAL_LABELS)

    chex_results[model_key] = {'auroc': auroc_dict, 'auprc': auprc_dict}

    print(f'  AUROC (macro): {auroc_dict["macro_avg"]:.4f}')
    print(f'  AUPRC (macro): {auprc_dict["macro_avg"]:.4f}')

In [ ]:
# ── 6. NIH Test Set 결과 로드 (비교 기준) ────────────────────────────────────
# 04_Training.ipynb 에서 저장한 y_prob/y_true를 활용
nih_results = {}

for model_key in ['densenet', 'efficientnet', 'vit']:
    prob_path = CHECKPOINT_DIR / f'{model_key}_y_prob.npy'
    true_path = CHECKPOINT_DIR / f'{model_key}_y_true.npy'
    if not prob_path.exists():
        print(f'[{model_key}] NIH 결과 없음 ({prob_path.name}), 건너뜀')
        continue
    y_prob_14 = np.load(prob_path)
    y_true_14 = np.load(true_path)
    y_prob_7  = y_prob_14[:, eval_indices]
    y_true_7  = y_true_14[:, eval_indices]
    nih_results[model_key] = {
        'auroc': compute_auroc(y_true_7, y_prob_7, EVAL_LABELS),
        'auprc': compute_auprc(y_true_7, y_prob_7, EVAL_LABELS),
    }
    print(f'[{model_key}] NIH AUROC (7-class macro): {nih_results[model_key]["auroc"]["macro_avg"]:.4f}')

In [ ]:
# ── 7. NIH vs CheXpert 비교 표 ───────────────────────────────────────────────
model_key = 'densenet'  # 비교할 모델 선택

if model_key in chex_results and model_key in nih_results:
    nih_auroc  = nih_results[model_key]['auroc']
    chex_auroc = chex_results[model_key]['auroc']
    gap        = compare_internal_external(nih_auroc, chex_auroc, EVAL_LABELS)

    rows = []
    for label in EVAL_LABELS:
        rows.append({
            'Disease':       label,
            'NIH AUROC':     round(nih_auroc.get(label, float('nan')), 4),
            'CheXpert AUROC':round(chex_auroc.get(label, float('nan')), 4),
            'Gap':           f"{gap.get(label, float('nan')):+.1%}",
        })
    rows.append({
        'Disease':        'macro_avg',
        'NIH AUROC':      round(nih_auroc['macro_avg'], 4),
        'CheXpert AUROC': round(chex_auroc['macro_avg'], 4),
        'Gap':            f"{gap['macro_avg']:+.1%}",
    })

    cmp_df = pd.DataFrame(rows)
    print(f'\n[Domain Shift 분석] {model_key}')
    print(cmp_df.to_string(index=False))
    cmp_df.to_csv(CHECKPOINT_DIR / f'{model_key}_domain_shift.csv', index=False)
else:
    print(f'{model_key} 결과가 없습니다.')

In [ ]:
# ── 8. 시각화 — NIH vs CheXpert AUROC Bar Chart ───────────────────────────────
if model_key in chex_results and model_key in nih_results:
    nih_vals  = [nih_auroc[l]  for l in EVAL_LABELS]
    chex_vals = [chex_auroc[l] for l in EVAL_LABELS]

    x     = range(len(EVAL_LABELS))
    width = 0.35
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Bar chart
    ax = axes[0]
    bars1 = ax.bar([i - width/2 for i in x], nih_vals,  width, label='NIH (Internal)',     color='steelblue')
    bars2 = ax.bar([i + width/2 for i in x], chex_vals, width, label='CheXpert (External)', color='coral')
    ax.set_xticks(list(x))
    ax.set_xticklabels(EVAL_LABELS, rotation=40, ha='right', fontsize=9)
    ax.set_ylim(0.5, 1.0)
    ax.set_ylabel('AUROC')
    ax.set_title(f'{model_key} — NIH vs CheXpert AUROC')
    ax.legend()
    ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5)

    # Gap chart
    ax2   = axes[1]
    gaps  = [chex_vals[i] - nih_vals[i] for i in range(len(EVAL_LABELS))]
    colors = ['red' if g < 0 else 'green' for g in gaps]
    ax2.bar(EVAL_LABELS, gaps, color=colors)
    ax2.axhline(0, color='black', linewidth=0.8)
    ax2.set_xticklabels(EVAL_LABELS, rotation=40, ha='right', fontsize=9)
    ax2.set_ylabel('AUROC Gap (CheXpert − NIH)')
    ax2.set_title('Domain Shift Gap')

    plt.tight_layout()
    fig_path = str(CHECKPOINT_DIR / f'{model_key}_domain_shift.png')
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print(f'차트 저장: {fig_path}')

In [ ]:
# ── 9. Domain Shift 원인 분석 요약 ────────────────────────────────────────────
print("""
[Domain Shift 주요 원인]

1. 촬영 장비 차이
   - NIH    : 30개 이상 다기관 데이터 (National Institutes of Health)
   - CheXpert: Stanford University Medical Center 단일 기관

2. 라벨링 방식 차이
   - NIH    : NLP 기반 자동 라벨링 (노이즈 존재)
   - CheXpert: 전문의 검토 + 불확실성 라벨 (-1)

3. 환자군 분포 차이
   - NIH    : 외래 환자 중심 (경증 多)
   - CheXpert: 입원/응급 포함, 중증도 높음

4. 촬영 프로토콜
   - NIH    : PA/AP 혼재
   - CheXpert: Frontal/Lateral 구분 명확, AP 비율 높음

[불확실성 전략별 예상 효과]
   u_zeros (현재) : Gap이 크게 나타남 (보수적)
   u_ones         : Gap이 작게 나타남 (낙관적)
   u_ignore       : 샘플 수 감소, 편향 제거

[권장 대응]
   1. Fine-tuning : CheXpert train split 일부로 추가 학습
   2. Domain Adaptation : Adversarial Training 적용
   3. u_ones 전략 비교 실험으로 라벨링 차이 영향 정량화
""")